# A2 · Random forest from scratch: the correlation between trees and the variance of the average

**Type**: Implement from scratch · **Level**: ●●● challenge · **Time**: about 60 minutes · **extended path**

## Objective

Build a forest by hand (trees with `max_features`) and measure, for `max_features` = all, `sqrt` and 2, the **mean correlation between the tree predictions**, the variance of a single tree and of the average, and the accuracy. Check the formula $\rho\sigma^2 + (1-\rho)\sigma^2/B$ and understand
why a forest of **worse** trees (with limited features) does better than bagging.

## Dataset

The same dataset as A1 (`make_classification`, 12 features): 60/40 split; the predictions are class 1 probabilities (`predict_proba`) of the trees on the test set.

## Educational purpose

Understand the real reason the random forest exists: it is not a better tree, it is a set of trees that are **more different from each other**. Each tree with a small `max_features` is less accurate, but the average gains more than each tree loses.

## Task

1. `forest(X, y, B, max_features, seed)`: `B` trees on bootstraps, each `DecisionTreeClassifier(max_features=max_features, random_state=b)` (with `max_features=None` = all the features). Return the list of trees.
2. For `B = 60` and `max_features` in `[None, "sqrt", 2]` build the matrix `P` (trees × test examples) of the predicted probabilities of class 1 and compute: `rho` (mean correlation between different pairs of trees: mean of the **off-diagonal** coefficients of `np.corrcoef(P)`), `single_var` (`P.var(axis=0).mean()`: variance over the trees of a prediction, averaged over the examples), `single_acc` (mean accuracy of a tree: threshold 0.5 on the rows of `P`) and `forest_acc` (accuracy of the mean `P.mean(0)` with threshold 0.5).
3. **The formula:** the variance of the average of `B` trees with variance `single_var` and correlation `rho` is `rho * single_var + (1 - rho) * single_var / B`: compute it and save it as `mean_var_formula`. Put everything in `table` (index `max_features` as a string: `"None"`, `"sqrt"`, `"2"`; columns `rho`, `single_acc`, `forest_acc`, `single_var`, `mean_var_formula`).
4. Save `best` (the `max_features` with the highest `forest_acc`) and `residual_variance_rho = rho * single_var` for that `max_features`: the term that **does not drop** when adding trees.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(2000, n_features=12, n_informative=5, n_redundant=2, class_sep=0.8, flip_y=0.08, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
B = 60
options = [None, "sqrt", 2]

## Your work

In [ ]:
def forest(X, y, B, max_features, seed=0):
    ...

In [ ]:
rows = {}
for mf in options:
    trees = forest(X_tr, y_tr, B, mf)
    P = np.array([t.predict_proba(X_te)[:, 1] for t in trees])         # (trees, examples)
    rows[str(mf)] = ...
table = pd.DataFrame(rows).T
table.round(4)

In [ ]:
best = None
residual_variance_rho = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`rho = (np.corrcoef(P).sum() - len(P)) / (len(P) * (len(P) - 1))`; `single_var = P.var(axis=0).mean()` (variance over the trees, averaged over the examples); `mean_var_formula = rho * single_var + (1 - rho) * single_var / B`. Accuracy: `((P > 0.5) == y_te).mean(axis=1).mean()` for the single tree, `((P.mean(0) > 0.5) == y_te).mean()` for the forest.'):
    check(list(table.columns) == ["rho", "single_acc", "forest_acc", "single_var", "mean_var_formula"] and list(table.index) == ["None", "sqrt", "2"], "table with three rows and five columns", "columns: rho, single_acc, forest_acc, single_var, mean_var_formula")
    check(table["rho"].is_monotonic_decreasing and table.loc["None", "rho"] > 0.45 and table.loc["2", "rho"] < 0.42, "the correlation between trees drops with max_features: from over 0.45 (all) to under 0.42 (2)", "mean of the off-diagonal coefficients of np.corrcoef(P)")
    check(table["single_acc"].is_monotonic_decreasing and table.loc["None", "single_acc"] - table.loc["2", "single_acc"] > 0.03, "a single tree with fewer features is worse (by over 3 points with max_features=2)", "mean accuracy of a tree")
    check(table.loc["sqrt", "forest_acc"] > table.loc["None", "forest_acc"] + 0.008 and table.loc["2", "forest_acc"] > table.loc["None", "forest_acc"], "but the forest with sqrt and with 2 features is better than bagging (max_features=all)", "forest_acc")
    check(best in ("sqrt", "2") and close(residual_variance_rho, table.loc[best, "rho"] * table.loc[best, "single_var"]), "the best is sqrt or 2; residual variance = rho * single variance", "the term that does not drop when adding trees")
    check((table["mean_var_formula"] < 0.6 * table["single_var"]).all() and table["mean_var_formula"].is_monotonic_decreasing, "the variance of the average is less than 60% of that of a tree (almost as much as rho), and drops with a smaller max_features", "rho * s2 + (1 - rho) * s2 / B")
done()

---
The commented solution is in `solutions/A_from_scratch/A2_why_max_features_decorrelates_the_trees.ipynb`. Open it after you pass the check, or after you have really tried.